# 1 · Calidad de datos

Antes de construir KPIs o modelos hay que saber **qué tan confiables son los datos** (Etapa 2 de la guía de
proyecto real: *exploración y análisis de datos*; "enfocarse en la calidad y cantidad de los datos").

Este notebook recorre las cuatro fuentes del LMS (trazabilidad, participación, calificaciones y matriculados):

1. Qué archivos hay y cuáles se usan.
2. Calidad dentro de cada tabla: nulos, duplicados, formatos, valores fuera de rango, atípicos.
3. **Consistencia entre fuentes**: ¿cuentan la misma historia?
4. Resumen de controles y qué decide el proyecto con cada hallazgo.

La limpieza tabla por tabla está en `src/limpieza.py` (y su reporte en `reports/calidad_datos.md`); los controles
entre fuentes que salen de este análisis quedaron como código en **`src/calidad.py`** (D69), así se vuelven a correr
cada vez que llega un archivo nuevo.

In [1]:
import os
from pathlib import Path
if not Path("src").exists():          # el notebook está en notebooks/: subo a la raíz
    os.chdir("..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from src import calidad, ingesta, metricas as m
from src.config import BRONZE, RAW, SILVER

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

bronze = {t: pd.read_parquet(BRONZE / f"{t}.parquet") for t in ["trazabilidad", "participacion", "calificaciones", "matriculados"]}
silver = {t: pd.read_parquet(SILVER / f"{t}.parquet") for t in bronze}
d = m.cargar()

## 1. Inventario de archivos

Cada Excel se reconoce por su **contenido** (tipo y materia), no por el nombre (D31). Las calificaciones y la
participación son "fotos": si hubiera dos del mismo instrumento, vale la más reciente (D32).

In [2]:
infos = ingesta.descubrir(RAW)
usados, reemplazados = ingesta.seleccionar(infos)
inv = pd.DataFrame([{"tipo": i.tipo, "materia": i.asignatura, "archivo": i.nombre, "exportado": i.fecha_exportacion,
                     "instrumento": i.instrumento, "errores": "; ".join(i.errores)} for i in infos])
print(f"{len(infos)} archivos en raw, {len(usados)} usados, {len(reemplazados)} reemplazados por uno más reciente")
inv.groupby(["tipo", "materia"]).size().unstack(fill_value=0)

26 archivos en raw, 26 usados, 0 reemplazados por uno más reciente


materia,Algoritmos y Estructuras de Datos,Probabilidad y Estadística,Álgebra
tipo,,,
calificaciones,5,9,3
matriculados,1,1,1
participacion,1,1,1
trazabilidad,1,1,1


In [3]:
inv[inv["tipo"].eq("calificaciones")].sort_values(["materia", "exportado"])[["materia", "instrumento", "exportado"]]

,materia,instrumento,exportado
0,Algoritmos y Estructuras de Datos,Autoevaluación - Módulo 1,2025-10-24 11:55:00
1,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 2,2025-10-24 11:55:00
2,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 3,2025-10-24 11:55:00
3,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 4,2025-10-24 11:55:00
4,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 5,2025-10-24 11:55:00
10,Probabilidad y Estadística,🧠 PARCIAL 1,2025-10-24 11:50:00
11,Probabilidad y Estadística,👉 Actividad Rápida 1,2025-10-24 11:50:00
12,Probabilidad y Estadística,👉 Actividad Rápida 2,2025-10-24 11:50:00
13,Probabilidad y Estadística,🚀Recuperatorio: Evaluación Semanal Nº1,2025-10-24 11:50:00
14,Probabilidad y Estadística,🚀Evaluación Semanal Nº1,2025-10-24 11:50:00


## 2. Calidad dentro de cada tabla

### 2.1 Filas de bronze a silver y valores faltantes

In [4]:
pd.DataFrame({"bronze": {t: len(x) for t, x in bronze.items()}, "silver": {t: len(x) for t, x in silver.items()}}) \
  .assign(quitadas=lambda t: t["bronze"] - t["silver"])

,bronze,silver,quitadas
trazabilidad,159804,159118,686
participacion,986,986,0
calificaciones,1793,1793,0
matriculados,516,513,3


In [5]:
nulos = []
for t, x in silver.items():
    for c in x.columns:
        s = x[c]
        faltan = s.isna() | s.astype(str).str.strip().eq("")
        if faltan.any():
            nulos.append({"tabla": t, "columna": c, "faltantes": int(faltan.sum()), "%": round(100 * faltan.mean(), 1)})
pd.DataFrame(nulos).sort_values(["tabla", "%"], ascending=[True, False])

,tabla,columna,faltantes,%
5,calificaciones,Detalle,1793,100.0
6,matriculados,Instancia,21,4.1
1,participacion,Email,986,100.0
3,participacion,Primera interacción,156,15.8
4,participacion,Última interacción,156,15.8
2,participacion,Avance,29,2.9
0,trazabilidad,Acción,19,0.0


### 2.2 Trazabilidad: duplicados, perfiles y calendario

In [6]:
tz_b, tz = bronze["trazabilidad"], silver["trazabilidad"]
dup = tz_b.duplicated().sum()
print(f"Duplicados exactos en bronze: {dup} ({100 * dup / len(tz_b):.2f} %) -> se eliminan en silver")
print(f"Filas sin 'Acción': {tz['Acción'].isna().sum() + tz['Acción'].astype(str).str.strip().eq('').sum()}")
tz.groupby("Perfil").agg(eventos=("Fecha", "size"), usuarios=("Usuario", "nunique"),
                         desde=("Fecha", "min"), hasta=("Fecha", "max"))

Duplicados exactos en bronze: 686 (0.43 %) -> se eliminan en silver
Filas sin 'Acción': 19


,eventos,usuarios,desde,hasta
Perfil,,,,
Alumnos,149458,285,2025-08-05 10:18:05,2025-10-24 11:53:27
Coordinadores,383,1,2025-06-04 14:19:41,2025-10-24 11:51:30
Tutores,9277,8,2025-06-06 19:30:37,2025-10-23 19:56:04


In [7]:
al = tz[tz["Perfil"].eq("Alumnos")]
por_dia = al.groupby([al["Fecha"].dt.normalize(), "asignatura"]).size().rename("eventos").reset_index()
fig = px.line(por_dia, x="Fecha", y="eventos", color="asignatura", title="Eventos de alumnos por día")
fig.update_layout(height=330, legend_title=None)
fig.show()
dias = pd.date_range(al["Fecha"].min().normalize(), al["Fecha"].max().normalize())
faltan = sorted(set(dias) - set(al["Fecha"].dt.normalize()))
print("Días de cursada sin ningún evento de alumnos:", [f"{x:%d/%m}" for x in faltan] or "ninguno")
print("Último evento:", al["Fecha"].max(), "(la exportación es del 24/10 a la mañana: ese día está incompleto)")

Días de cursada sin ningún evento de alumnos: ninguno
Último evento: 2025-10-24 11:53:27 (la exportación es del 24/10 a la mañana: ese día está incompleto)


In [8]:
hora = al["Fecha"].dt.hour.value_counts().sort_index()
fig = px.bar(x=hora.index, y=hora.values, labels={"x": "Hora", "y": "Eventos"}, title="Eventos de alumnos por hora")
fig.update_layout(height=280)
fig.show()

### 2.3 Atípicos en la actividad

Se marca como atípico a quien supera Q3 + 1,5 × IQR de eventos en su materia. No se borran: un alumno que entra
mucho no es un error, pero conviene saber cuántos hay porque arrastran los promedios (por eso los dashboards usan
medianas y Spearman, D38).

In [9]:
e = al.groupby(["asignatura", "Usuario"]).size().rename("eventos").reset_index()
def iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return q3 + 1.5 * (q3 - q1)
e["tope"] = e.groupby("asignatura")["eventos"].transform(iqr)
print(e.groupby("asignatura").apply(lambda g: pd.Series({"alumnos": len(g), "mediana": g["eventos"].median(),
      "tope_IQR": round(g["tope"].iloc[0]), "atipicos": int((g["eventos"] > g["tope"]).sum()),
      "maximo": g["eventos"].max()}), include_groups=False))
fig = px.box(e, x="asignatura", y="eventos", points="outliers", title="Eventos por alumno y materia")
fig.update_layout(height=320)
fig.show()

                                   alumnos  mediana  tope_IQR  atipicos  maximo
asignatura                                                                     
Algoritmos y Estructuras de Datos    276.0    141.0     506.0       6.0   818.0
Probabilidad y Estadística           273.0    199.0     754.0       7.0   986.0
Álgebra                              268.0    139.5     494.0       3.0   828.0


In [10]:
s = d.sesiones[d.sesiones["en_base"]]
print(s["duracion_min"].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).round(1))
print(f"Sesiones de menos de 1 minuto: {100 * (s['duracion_min'] < 1).mean():.0f} %  "
      "(una sesión de un solo evento dura 0: los minutos subestiman el tiempo real)")

count    19755.0
mean         7.1
std         14.1
min          0.0
50%          0.9
75%          7.2
90%         23.8
99%         66.7
max        218.0
Name: duracion_min, dtype: float64
Sesiones de menos de 1 minuto: 51 %  (una sesión de un solo evento dura 0: los minutos subestiman el tiempo real)


### 2.4 Calificaciones

`Nota` viene como texto `"24.00/25.00"`: el denominador es el puntaje máximo del instrumento. Se controla que
el máximo sea siempre el mismo dentro de cada instrumento y que ninguna nota quede fuera de rango.

In [11]:
ca = silver["calificaciones"]
print("Máximos distintos por instrumento:", ca.groupby("instrumento")["Nota máxima"].nunique().max())
print("Notas fuera de rango (< 0 o > máximo):", int(((ca["Nota obtenida"] < 0) | (ca["Nota obtenida"] > ca["Nota máxima"])).sum()))
print("Duplicados alumno × instrumento:", int(ca.duplicated(["Usuario", "instrumento"]).sum()))
fig = px.box(ca, x="instrumento", y="Nota (%)", color="asignatura", title="Nota (%) por instrumento (solo quienes rindieron)")
fig.update_layout(height=380, xaxis_title=None, legend_title=None)
fig.show()

Máximos distintos por instrumento: 1
Notas fuera de rango (< 0 o > máximo): 0
Duplicados alumno × instrumento: 0


**¿Sirve la columna `Observaciones` (aprobada / no aprobada)?** Si fuera el corte de la materia (60 %), los
"aprobada" deberían estar todos arriba de 60.

In [12]:
obs = calidad.observaciones_vs_nota()
obs[["asignatura", "instrumento", "min_aprobada", "max_aprobada", "min_no_aprobada", "max_no_aprobada"]]

,asignatura,instrumento,min_aprobada,max_aprobada,min_no_aprobada,max_no_aprobada
0,Algoritmos y Estructuras de Datos,Autoevaluación - Módulo 1,60.00,100.00,50.0,50.00
1,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 2,60.00,100.00,NaN,NaN
2,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 3,30.00,100.00,NaN,NaN
3,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 4,70.00,100.00,NaN,NaN
4,Algoritmos y Estructuras de Datos,Autoevaluación - Unidad 5,70.00,100.00,NaN,NaN
5,Probabilidad y Estadística,👉 Actividad Rápida 1,66.67,100.00,NaN,NaN
6,Probabilidad y Estadística,👉 Actividad Rápida 2,100.00,100.00,0.0,50.00
7,Probabilidad y Estadística,👉 Actividad Rápida 3,75.00,100.00,0.0,50.00
8,Probabilidad y Estadística,👉 Actividad Rápida 4,87.50,100.00,0.0,75.00
9,Probabilidad y Estadística,👉 Actividad Rápida 5,66.67,100.00,0.0,33.33


**¿Cada nota tiene su rendición en la trazabilidad?** Las notas se fechan con el evento "Realizar evaluación"
del alumno (D64), así que importa cuántas no lo tienen.

In [13]:
sin = calidad.notas_sin_rendicion(d)
print(f"{len(sin)} notas sin evento de rendición, de {int(d.calificaciones['rindio'].sum())}")
sin

8 notas sin evento de rendición, de 1793


,asignatura,id_alumno,id_instrumento,instrumento,nota_pct
558,Probabilidad y Estadística,ALU_1044,PYE-05,🚀Evaluación Semanal Nº1,83.33
1057,Probabilidad y Estadística,ALU_1220,PYE-05,🚀Evaluación Semanal Nº1,0.00
1063,Probabilidad y Estadística,ALU_1221,PYE-05,🚀Evaluación Semanal Nº1,0.00
1099,Probabilidad y Estadística,ALU_1235,PYE-05,🚀Evaluación Semanal Nº1,88.89
1235,Probabilidad y Estadística,ALU_1296,PYE-05,🚀Evaluación Semanal Nº1,0.00
1445,Álgebra,ALU_1101,ALG-03,1° Parcial,0.00
1482,Álgebra,ALU_1122,ALG-01,AUTOEVALUACION1,80.00
1616,Álgebra,ALU_1195,ALG-03,1° Parcial,68.00


### 2.5 Participación

In [14]:
pa = silver["participacion"]
print(pa["Perfil"].value_counts().to_dict())
pal = pa[pa["Perfil"].eq("Alumnos")]
print("Alumnos por materia en el reporte:", pal["asignatura"].value_counts().to_dict())
print("Con 0 interacciones:", int(pd.to_numeric(pal["Interacciones"], errors="coerce").eq(0).sum()))
print("Columnas vacías en todas las filas:", [c for c in pa.columns if pa[c].isna().all() or pa[c].astype(str).str.strip().eq("").all()])

{'Alumnos': 957, 'Invitados': 15, 'Tutores': 11, 'Coordinadores': 3}
Alumnos por materia en el reporte: {'Algoritmos y Estructuras de Datos': 319, 'Probabilidad y Estadística': 319, 'Álgebra': 319}
Con 0 interacciones: 140
Columnas vacías en todas las filas: ['Email']


### 2.6 Matriculados

In [15]:
ma = silver["matriculados"]
print(ma["Estado Insc."].value_counts().to_dict(), "| comisiones:", ma["Comisión"].value_counts().to_dict())
varias = ma.groupby(["asignatura", "Identificación"]).size()
print("Alumnos con más de una fila (altas y bajas):", int((varias > 1).sum()), "-> vale el último estado (D2)")

{'Aceptada': 492, 'Baja': 21} | comisiones: {'2320': 175, '2321': 171, '2319': 167}
Alumnos con más de una fila (altas y bajas): 11 -> vale el último estado (D2)


## 3. Consistencia entre fuentes

### 3.1 ¿Están los mismos alumnos?

In [16]:
calidad.cobertura_ids()

,fuente,alumnos,no estan en participacion,no estan en calificaciones,no estan en matriculados,no estan en trazabilidad
0,trazabilidad,285,0.0,68.0,118.0,0.0
1,participacion,319,0.0,102.0,151.0,34.0
2,calificaciones,217,0.0,0.0,62.0,0.0
3,matriculados,168,0.0,13.0,0.0,1.0


In [17]:
b = m.base(d)
print("Base por materia (inscriptos ∪ con nota, D2):")
b.groupby("asignatura").agg(base=("id_alumno", "size"), inscriptos=("inscripto", "sum"),
                            no_inscriptos=("inscripto", lambda s: int((~s).sum())))

Base por materia (inscriptos ∪ con nota, D2):


,base,inscriptos,no_inscriptos
asignatura,,,
Algoritmos y Estructuras de Datos,204,165,39
Probabilidad y Estadística,216,164,52
Álgebra,217,163,54


### 3.2 Participación contra trazabilidad

El reporte de participación trae "Interacciones" por alumno; la trazabilidad, un evento por fila. Si midieran
lo mismo deberían coincidir.

In [18]:
pv = calidad.participacion_vs_trazabilidad()
pv1 = pv[pv["trazabilidad"] > 0]
print(f"Iguales: {100 * (pv1['participacion'] == pv1['trazabilidad']).mean():.1f} %  |  "
      f"cociente mediano participación / trazabilidad: {pv1['cociente'].median():.2f}  |  "
      f"Spearman: {pv1[['participacion', 'trazabilidad']].corr(method='spearman').iloc[0, 1]:.3f}")
fig = px.scatter(pv1, x="trazabilidad", y="participacion", color="asignatura", log_x=True, log_y=True, opacity=0.6,
                 title="Interacciones: reporte de participación vs. eventos de trazabilidad (escala log)")
fig.add_shape(type="line", x0=1, y0=1, x1=1500, y1=1500, line=dict(dash="dash", color="gray"))
fig.update_layout(height=380, legend_title=None)
fig.show()

Iguales: 1.7 %  |  cociente mediano participación / trazabilidad: 1.26  |  Spearman: 0.994


In [19]:
m.concordancia_tiempo(d)

,asignatura,rho,p_valor,n,minutos_total,tiempo_uso_total
0,Algoritmos y Estructuras de Datos,0.997,3.611129e-232,204,39440.3,39957
1,Probabilidad y Estadística,0.971,1.383641e-135,216,64021.4,82253
2,Álgebra,0.947,1.212869e-107,217,36946.3,47469


### 3.3 Matrícula contra actividad

In [20]:
ins = calidad.inscripcion_vs_actividad(d)
print(f"Inscriptos que entraron antes de inscribirse: {(ins['dias_inscripcion_menos_ingreso'] > 0).sum()} de {len(ins)}")
fig = px.histogram(ins, x="dias_inscripcion_menos_ingreso", color="asignatura", nbins=40,
                   title="Días entre el primer ingreso al aula y la inscripción (positivo = se inscribió después)")
fig.update_layout(height=320, legend_title=None, bargap=0.05)
fig.show()
bajas = d.matricula[d.matricula["estado"].eq("Baja")]
bajas[["asignatura", "id_alumno", "fecha_inscripcion", "ultima_interaccion", "en_base"]]

Inscriptos que entraron antes de inscribirse: 290 de 492


,asignatura,id_alumno,fecha_inscripcion,ultima_interaccion,en_base
53,Algoritmos y Estructuras de Datos,ALU_1084,2025-09-02 22:58:46,2025-08-17 11:32:26,False
134,Algoritmos y Estructuras de Datos,ALU_1193,2025-08-04 17:01:36,2025-08-06 09:28:58,False
263,Probabilidad y Estadística,ALU_1084,2025-08-08 13:21:54,2025-08-17 11:31:31,False
319,Probabilidad y Estadística,ALU_1154,2025-08-04 19:16:26,2025-10-10 10:46:57,False
349,Probabilidad y Estadística,ALU_1193,2025-08-04 17:01:57,2025-10-03 09:38:20,False
482,Álgebra,ALU_1084,2025-08-08 13:21:55,2025-08-22 18:05:45,True
567,Álgebra,ALU_1193,2025-08-04 17:02:02,2025-10-11 14:27:24,True


In [21]:
ips = calidad.ips_compartidas()
print(f"{len(ips)} IPs con más de un alumno; las 5 con más: {ips.head().tolist()}")

139 IPs con más de un alumno; las 5 con más: [17, 6, 6, 5, 5]


## 4. Resumen de controles

La misma tabla sale de `python -m src.calidad`.

In [22]:
calidad.chequeos(d)

,area,control,resultado,valor,decision
0,Identidad,Alumnos del reporte de participación sin ningún evento en la trazabilidad,Atención,34 de 319,Participación lista a toda la cohorte (319); la base de cada materia es insc...
1,Identidad,Alumnos con nota que nunca entraron al aula,OK,0,"Si hubiera, la nota no tendría actividad que la explique."
2,Identidad,Matriculados que nunca entraron al aula,Atención,1,Quedan en la base con todo en 0 (Libre) y alimentan la alerta de inscriptos ...
3,Identidad,IDs con más de un nombre entre fuentes,OK,0,El ID es la clave: los nombres solo se muestran.
4,Notas,'Observaciones' no coincide con el corte de 60 % de la nota,Atención,54 de 1793 notas (cada instrumento tiene su propio criterio),No se usa Observaciones: la condición sale de la nota y los cortes de ESQUEM...
5,Notas,Puntaje máximo distinto dentro de un mismo instrumento,OK,1 máximo(s) por instrumento,El máximo del instrumento sale del denominador de la nota.
6,Notas,Notas sin el evento 'Realizar evaluación' del alumno,Atención,8 de 1793,Para fechar esas notas se usa la última realización del instrumento (D64).
7,Actividad,Interacciones del reporte de participación contra eventos de la trazabilidad,Atención,"iguales en 2 %, cociente mediano 1.26, ρ = 0.99",Miden lo mismo con otra regla de conteo: la actividad se calcula siempre con...
8,Actividad,Eventos anteriores al primer ingreso de un alumno,OK,1739 (docentes armando el aula desde el 04/06),Son de Tutores y Coordinadores: van a otros_perfiles_interaccion (D4).
9,Actividad,Sesiones de menos de 1 minuto,OK,51 % (mediana 0.9 min),Una sesión de un solo evento dura 0: los minutos subestiman el tiempo real.


## Conclusiones

**Los datos son utilizables y consistentes en lo esencial**, pero hay cosas que hay que tener en cuenta al leer
los indicadores. Cada hallazgo tiene su decisión:

| Hallazgo | Qué decide el proyecto |
|---|---|
| 26 archivos, todos reconocidos por su contenido; ninguno reemplazado. | La ingesta no depende del nombre del archivo (D31-D33). |
| 686 duplicados exactos en la trazabilidad (0,43 %) y 19 filas sin acción. | Los duplicados se borran; las 19 filas se conservan (son descargas en foros). |
| `Email` y `Detalle` vienen vacíos en todas las filas (anonimización). | Se conservan las columnas, no se usan. |
| Los docentes arman el aula desde junio; los alumnos entran desde el 05/08. | Los perfiles no alumno van a tablas aparte (D4); la cursada empieza con el primer evento de un alumno. |
| No hay días sin datos en la cursada; el 24/10 está incompleto (la exportación es de esa mañana). | La fecha de corte por defecto es el 24/10 (D29); las métricas "de la última semana" de ese día subestiman. |
| La mitad de las sesiones dura menos de 1 minuto y unos pocos alumnos tienen 3 a 5 veces la actividad de la mediana. | Se usan medianas y Spearman (D38) y log en los modelos; los atípicos no se borran: no son errores. |
| Las notas están en rango, sin duplicados y con un solo puntaje máximo por instrumento. | La nota se normaliza con ese máximo (D16-D17). |
| `Observaciones` usa un criterio propio de cada instrumento: hay "aprobada" con 12 % y "no aprobada" con 75 %. | **No se usa**: la condición sale de la nota y de los cortes de cada materia. |
| 8 notas no tienen su evento "Realizar evaluación". | Para fecharlas se usa la última realización del instrumento (D64). |
| El reporte de participación lista a los 319 alumnos de la cohorte en las tres materias (34 nunca entraron). | La base de cada materia es inscriptos ∪ con nota (D2), no el reporte. |
| Participación cuenta un 26 % más de interacciones que la trazabilidad, pero ordena igual (ρ = 0,99); el tiempo estimado también concuerda con los minutos de las sesiones (ρ 0,95-1,00). | La actividad se calcula siempre con la trazabilidad (es la que tiene fecha); el reporte sirve de control. |
| 290 de 492 inscriptos entraron al aula antes de su fecha de inscripción. | La fecha de inscripción no marca el inicio de nadie: se usa el primer evento. |
| Hay bajas con actividad posterior y alumnos con nota que no figuran en Matriculados. | La baja formal no se usa como abandono (D67); los no inscriptos con actividad son un KPI aparte (D2). |
| Hay IPs compartidas por hasta 17 alumnos. | La IP no identifica a nadie (redes compartidas): no se usa. |

**Qué quedó en código:** los controles entre fuentes de la sección 4 están en `src/calidad.py`
(`python -m src.calidad`), con sus tests en `tests/test_calidad.py`, para volver a correrlos cuando llegue un
archivo nuevo.